# 23 · Batched LLM rollouts: left padding, position ids, KV cache, response masks, log-prob recompute

**Needed for:** every LLM RL implementation (RL track, steps R3–R7). Most silent bugs in RLHF/GRPO code live here,
not in the loss.

An RL step for an LLM:
1. **Generate** responses for a batch of prompts of different lengths (batched, with a KV cache).
2. **Mask** everything that isn't a real response token (prompt, padding, tokens after end-of-sequence).
3. **Recompute** log-probs of the generated tokens with a normal forward pass (for the policy and the reference).
4. Compute rewards, advantages and the loss (notebooks 20–22).

The test for steps 1–3: batched, padded generation must give the same tokens and log-probs as generating each
prompt alone.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)


def drill(test, fn):
    """Run a test against your own implementation; prints TODO until you write it."""
    try:
        test(fn)
    except NotImplementedError:
        print(f"TODO  {fn.__name__}")
    else:
        print(f"PASS  {fn.__name__}")

## 1. Left padding, attention masks, position ids

For generation, pad on the **left** so every sequence's last real token sits in the same final column and new
tokens append in lockstep. Two consequences:
- The attention mask must hide pad tokens as keys.
- **Position ids** must count only real tokens: `cumsum(mask) - 1`. With left padding, `arange(T)` is wrong: it
  shifts every position of shorter prompts, which changes the output of any model with positional information.

In [ ]:
PAD, EOS = 0, 1


def left_pad(prompts):
    """list of 1-D LongTensors -> (ids (B, T), attention_mask (B, T))."""
    T = max(len(p) for p in prompts)
    ids = torch.full((len(prompts), T), PAD, dtype=torch.long)
    mask = torch.zeros(len(prompts), T, dtype=torch.long)
    for i, p in enumerate(prompts):
        ids[i, T - len(p):] = p
        mask[i, T - len(p):] = 1
    return ids, mask


def position_ids_from_mask(mask):
    return (mask.cumsum(-1) - 1).clamp(min=0)


def test_left_pad(fn):
    ids, mask = fn([torch.tensor([5, 6, 7]), torch.tensor([8])])
    assert ids.tolist() == [[5, 6, 7], [PAD, PAD, 8]]
    assert mask.tolist() == [[1, 1, 1], [0, 0, 1]]
    assert position_ids_from_mask(mask).tolist() == [[0, 1, 2], [0, 0, 0]]


test_left_pad(left_pad)
print("left padding ok")

## 2. A tiny GPT with a KV cache that respects padding

`forward(ids, attn_mask, pos_ids, cache)`: `ids` are the *new* tokens, `attn_mask` covers *all* tokens so far
(cached + new). The attention mask is causal **and** hides pad keys. Each position may always attend to itself, so
pad rows never have an empty softmax (their outputs are garbage but never used).

In [ ]:
class TinyGPT(nn.Module):
    def __init__(self, vocab=50, d=64, heads=4, layers=2, max_pos=64):
        super().__init__()
        self.h = heads
        self.tok, self.pos = nn.Embedding(vocab, d), nn.Embedding(max_pos, d)
        self.qkv = nn.ModuleList(nn.Linear(d, 3 * d) for _ in range(layers))
        self.proj = nn.ModuleList(nn.Linear(d, d) for _ in range(layers))
        self.mlp = nn.ModuleList(nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d)) for _ in range(layers))
        self.ln1 = nn.ModuleList(nn.LayerNorm(d) for _ in range(layers))
        self.ln2 = nn.ModuleList(nn.LayerNorm(d) for _ in range(layers))
        self.ln_f, self.head = nn.LayerNorm(d), nn.Linear(d, vocab)

    def forward(self, ids, attn_mask, pos_ids, cache=None):
        B, t = ids.shape
        x = self.tok(ids) + self.pos(pos_ids)
        past = 0 if cache is None else cache[0][0].shape[2]
        S = past + t
        q_idx = torch.arange(past, S)[:, None]
        k_idx = torch.arange(S)[None, :]
        allowed = (k_idx <= q_idx)[None, None] & attn_mask[:, None, None, :].bool()   # causal & not pad
        allowed = allowed | (k_idx == q_idx)[None, None]                                # always see yourself
        new_cache = []
        for i in range(len(self.qkv)):
            q, k, v = self.qkv[i](self.ln1[i](x)).split(x.shape[-1], -1)
            split = lambda z: z.view(B, t, self.h, -1).transpose(1, 2)
            q, k, v = split(q), split(k), split(v)
            if cache is not None:
                k, v = torch.cat([cache[i][0], k], 2), torch.cat([cache[i][1], v], 2)
            new_cache.append((k, v))
            a = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed)
            x = x + self.proj[i](a.transpose(1, 2).reshape(B, t, -1))
            x = x + self.mlp[i](self.ln2[i](x))
        return self.head(self.ln_f(x)), new_cache

## 3. Greedy generation: batched vs one at a time

Stop a sequence after it emits EOS; keep padding its row with PAD so the batch stays rectangular.
Everything runs in float64 so that tiny numeric differences between padded and unpadded matmuls can't flip an
argmax and hide a real bug.

In [ ]:
@torch.no_grad()
def generate(model, prompts, max_new=8):
    ids, mask = left_pad(prompts)
    pos = position_ids_from_mask(mask)
    logits, cache = model(ids, mask, pos)
    out, logps = [], []
    finished = torch.zeros(len(prompts), dtype=torch.bool)
    next_pos = pos[:, -1] + 1
    for _ in range(max_new):
        step_logp = logits[:, -1].log_softmax(-1)
        tok = step_logp[:, 1:].argmax(-1) + 1                  # never emit PAD (id 0) as a real token
        tok = torch.where(finished, torch.full_like(tok, PAD), tok)
        out.append(tok)
        logps.append(step_logp.gather(-1, tok[:, None]).squeeze(-1))
        finished |= tok == EOS
        mask = torch.cat([mask, torch.ones(len(prompts), 1, dtype=torch.long)], 1)
        logits, cache = model(tok[:, None], mask, next_pos[:, None], cache)
        next_pos = next_pos + 1
    return torch.stack(out, 1), torch.stack(logps, 1)


torch.manual_seed(0)
model = TinyGPT().double().eval()
prompts = [torch.randint(2, 50, (n,)) for n in (7, 3, 5)]
batched_tokens, batched_logps = generate(model, prompts)
for i, p in enumerate(prompts):
    solo_tokens, solo_logps = generate(model, [p])
    n = len(solo_tokens[0])
    assert torch.equal(batched_tokens[i, :n], solo_tokens[0]), (i, batched_tokens[i], solo_tokens[0])
    torch.testing.assert_close(batched_logps[i], solo_logps[0])
print("batched left-padded generation == generating each prompt alone")
print(batched_tokens)

**Try breaking it:** in `generate`, replace `position_ids_from_mask(mask)` with
`torch.arange(ids.shape[1]).expand_as(ids)` and rerun. The assertion fails for the shorter prompts: this exact bug
has shipped in real RL code.

## 4. Response mask: stop at the first EOS

The loss must include the EOS token itself (the model should learn when to stop) but nothing after it.

In [ ]:
def response_mask(tokens):
    """tokens: (B, L) generated tokens. 1 up to and including the first EOS, 0 after."""
    is_eos = (tokens == EOS).long()
    eos_before = is_eos.cumsum(-1) - is_eos           # number of EOS strictly before each position
    return (eos_before == 0).long()


def test_response_mask(fn):
    t = torch.tensor([[5, 6, EOS, PAD, PAD], [7, 8, 9, 10, 11], [EOS, PAD, PAD, PAD, PAD], [4, EOS, 3, EOS, 2]])
    assert fn(t).tolist() == [[1, 1, 1, 0, 0], [1, 1, 1, 1, 1], [1, 0, 0, 0, 0], [1, 1, 0, 0, 0]]


test_response_mask(response_mask)
print("response mask ok")

## 5. Recompute log-probs with one forward pass

Training needs the log-probs of the generated tokens *with gradients*, from one forward pass over prompt +
response (notebook 21's shift-by-one). They must match what generation saw, token for token. When they don't
(bf16 vs fp32, different attention kernels in vLLM vs the trainer), the "on-policy" ratio isn't 1 at the start,
and recent RL frameworks correct for this mismatch with importance weights.

In [ ]:
def recompute_logprobs(model, prompts, gen_tokens):
    ids, mask = left_pad(prompts)
    full = torch.cat([ids, gen_tokens], 1)
    full_mask = torch.cat([mask, torch.ones_like(gen_tokens)], 1)
    logits, _ = model(full, full_mask, position_ids_from_mask(full_mask))
    P = ids.shape[1]
    logp = logits[:, P - 1:-1].log_softmax(-1)                       # predictions for the generated positions
    return logp.gather(-1, gen_tokens[..., None]).squeeze(-1)


recomputed = recompute_logprobs(model, prompts, batched_tokens)
m = response_mask(batched_tokens).bool()
torch.testing.assert_close(recomputed[m], batched_logps[m])
print("recomputed log-probs match generation-time log-probs on all response tokens")

model_bf16 = TinyGPT().eval()
model_bf16.load_state_dict({k: v.float() for k, v in model.state_dict().items()})
with torch.autocast("cpu", dtype=torch.bfloat16):
    lp_bf16 = recompute_logprobs(model_bf16, prompts, batched_tokens).float()
ratio = torch.exp(lp_bf16[m] - batched_logps[m].float())
print(f"same weights, bf16 forward: importance ratio range [{ratio.min():.3f}, {ratio.max():.3f}] instead of exactly 1")

## Things to be able to explain

- Why right-pad for training but left-pad for generation?
- After EOS, why keep feeding PAD tokens instead of removing finished rows? (Rectangular batches; real engines like
  vLLM do remove them with continuous batching.)
- Where do generation-time and training-time log-probs diverge in practice, and why does that matter for PPO/GRPO?

## Drill

In [ ]:
def my_left_pad(prompts):
    raise NotImplementedError


def my_response_mask(tokens):
    raise NotImplementedError


drill(test_left_pad, my_left_pad)
drill(test_response_mask, my_response_mask)